# E1–E2: LoRA Fine-Tuning and Rank Sweep — CheXpert

E1 = reproduce standard LoRA (r=48). E2 = rank sweep over [4, 8, 16, 32, 48] to evaluate adaptation capacity scaling.

In [3]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if os.path.basename(os.getcwd()) != "DinoV2_PEFT":
    if not os.path.exists("DinoV2_PEFT"):
        subprocess.run(["git", "clone", "https://github.com/RudhanLodhi/DinoV2_PEFT.git"], check=True)
    os.chdir("DinoV2_PEFT")

subprocess.run(["pip", "install", "-r", "requirements.txt", "-q"], check=True)

from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint directory: /content/drive/MyDrive/dinov2_peft


In [11]:
!git pull

Updating 5907332..d84ad11
error: The following untracked working tree files would be overwritten by merge:
	src/models/__pycache__/peft_wrapper.cpython-313.pyc
Please move or remove them before you merge.
Aborting


## 1. Configuration

Load CheXpert benchmark dataset configuration and default LoRA adapter parameters.

In [4]:
"""
Load dataset configuration and default LoRA fine-tuning hyperparameters.
"""
from typing import Any
from src.utils.config import load_config

cfg: dict[str, Any] = load_config("configs/chexpert.yaml")
lora_cfg: dict[str, Any] = load_config("configs/peft/lora.yaml")
print("CheXpert Configuration:", cfg["dataset"])
print("Default LoRA Configuration:", lora_cfg["lora"])

CheXpert Configuration: {'name': 'chexpert', 'hf_repo': 'danjacobellis/chexpert', 'image_column': 'image', 'label_names': ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion'], 'num_labels': 5, 'hf_4class_encoding': True, 'uncertainty_policy': 'u_zeros', 'frontal_only': True}
Default LoRA Configuration: {'r': 48, 'lora_alpha': 16, 'lora_dropout': 0.1, 'bias': 'none', 'target_modules': ['query', 'value'], 'modules_to_save': ['classifier'], 'task_type': 'FEATURE_EXTRACTION'}


## 2. Build Datasets

Construct training and validation dataloaders with standard CheXpert normalization and augmentation transforms.

In [5]:
"""
Construct PyTorch DataLoaders for CheXpert train/val/test splits.
"""
from src.data import build_chexpert_dataloaders, build_transforms

pre = cfg["preprocessing"]
train_tf = build_transforms(
    pre["image_size"], pre["normalize_mean"], pre["normalize_std"], augment=True,
    crop_scale_min=pre.get("crop_scale_min", 0.75), crop_scale_max=pre.get("crop_scale_max", 1.0),
)
val_tf = build_transforms(
    pre["image_size"], pre["normalize_mean"], pre["normalize_std"], augment=False,
    resize_ratio=pre.get("resize_ratio", 1.14),
)
train_loader, val_loader, test_loader = build_chexpert_dataloaders(cfg, train_tf, val_tf)
NUM_LABELS: int = int(cfg["dataset"]["num_labels"])
print(
    f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, "
    f"Test batches: {len(test_loader)}"
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/4.21k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

data/train-00000-of-00023.parquet: reconstructing file:   0%|          |  0.00B /  487MB            

data/train-00000-of-00023.parquet: downloading bytes:           |  0.00B            

... (48 outputs left out)

Loading dataset shards:   0%|          | 0/23 [00:00<?, ?it/s]

Filter:   0%|          | 0/223414 [00:00<?, ? examples/s]

Filter:   0%|          | 0/234 [00:00<?, ? examples/s]

Train batches: 62, Val batches: 2, Test batches: 16


## 3. LoRA Model Construction

Define a model builder that applies PEFT LoRA adapters to the DINOv2 ViT-B/14 backbone and pairs it with a multi-label classification head.

In [6]:
"""
Factory function to construct LoRA-adapted DINOv2 model with multi-label classification head.
"""
from typing import Any
import torch
import torch.nn as nn
from src.models import apply_lora, count_trainable_parameters, load_backbone, MultiLabelHead
from src.utils.config import load_config

class LoRAModel(nn.Module):
    """
    End-to-end model combining LoRA-adapted DINOv2 backbone and multi-label classification head.
    """
    def __init__(self, backbone: nn.Module, head: nn.Module) -> None:
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Forward pass extracting CLS token and returning multi-label logits.
        """
        outputs = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.head(cls_token)

def build_lora_model(lora_overrides: dict[str, Any] | None = None) -> LoRAModel:
    """
    Builds a LoRA-wrapped DINOv2 model with a multi-label classification head.
    Returns the full model with PEFT adapter applied to the backbone.
    """
    lora_config: dict[str, Any] = load_config("configs/peft/lora.yaml", overrides=lora_overrides)
    backbone, _ = load_backbone()
    leaf_names = {name.rsplit(".", 1)[-1] for name, _ in backbone.named_modules() if name}
    attn_map = {"query": "q_proj", "key": "k_proj", "value": "v_proj"}
    lora_params = dict(lora_config["lora"])
    lora_params["target_modules"] = [
        attn_map[t] if t not in leaf_names and attn_map.get(t) in leaf_names else t
        for t in lora_params.get("target_modules", ["query", "value"])
    ]
    backbone = apply_lora(backbone, lora_params)
    head = MultiLabelHead(in_features=768, num_labels=NUM_LABELS)
    return LoRAModel(backbone, head)

## 4. E1 — Standard LoRA (r=48)

Train and evaluate standard LoRA with rank 48 matching the base paper's PEFT configuration.

In [7]:
"""
Train standard LoRA model (r=48), evaluate validation AUROC, and persist checkpoint.
"""
from typing import Any
import torch
from torch.optim import AdamW
from src.evaluation import Evaluator
from src.training.losses import MultiLabelBCELoss
from src.utils.logging_utils import ExperimentLogger

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_e1: LoRAModel = build_lora_model({"lora": {"r": 48, "lora_alpha": 16}})
model_e1.to(device)

trainable_e1, total_e1 = count_trainable_parameters(model_e1)
print(f"E1 Trainable parameters: {trainable_e1:,} / {total_e1:,} ({100 * trainable_e1 / total_e1:.2f}%)")

criterion = MultiLabelBCELoss()
optimizer = AdamW(
    [
        {"params": [p for n, p in model_e1.backbone.named_parameters() if p.requires_grad], "lr": 2e-4},
        {"params": model_e1.head.parameters(), "lr": 5e-3},
    ],
    weight_decay=1e-4,
)

epochs: int = 10
for epoch in range(epochs):
    model_e1.train()
    running_loss: float = 0.0
    for batch in train_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        logits = model_e1(pixel_values)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch [{epoch + 1:02d}/{epochs:02d}] - Loss: {running_loss / len(train_loader):.4f}")

evaluator = Evaluator(task="multilabel", label_names=cfg["dataset"]["label_names"])
results_e1: dict[str, Any] = evaluator.evaluate(model_e1, val_loader, device=str(device), split="val")
test_e1: dict[str, Any] = evaluator.evaluate(model_e1, test_loader, device=str(device), split="test")
print(f"E1 (r=48) Val Mean AUROC:  {results_e1['mean_auroc']:.4f}")
print(f"E1 (r=48) Test Mean AUROC: {test_e1['mean_auroc']:.4f}")

logger = ExperimentLogger()
logger.log(
    run_id="lora_r48_chexpert",
    dataset="CheXpert",
    backbone="DINOv2 ViT-B/14",
    method="LoRA",
    rank=48,
    trainable_params=trainable_e1,
    val_auroc=results_e1["mean_auroc"],
    test_auroc=test_e1["mean_auroc"],
)

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

NoMatchingPeftModuleError: Target modules {'query', 'value'} not found in the base model. Please check the target modules and try again.

## 5. E2 — Rank Sweep

Execute an empirical rank sweep across ranks [4, 8, 16, 32, 48] keeping alpha=16 and tracking trainable capacity versus AUROC.

In [ ]:
"""
Execute controlled LoRA rank sweep over [4, 8, 16, 32, 48] and record validation performance.
"""
from typing import Any
import pandas as pd
from torch.optim import AdamW

RANK_SWEEP: list[int] = [4, 8, 16, 32, 48]
sweep_results: list[dict[str, Any]] = []

for r in RANK_SWEEP:
    print(f"\n=== Running Rank Sweep: r={r} ===")
    model_r: LoRAModel = build_lora_model({"lora": {"r": r, "lora_alpha": 16}})
    model_r.to(device)
    trainable, total = count_trainable_parameters(model_r)

    opt = AdamW(
        [
            {"params": [p for n, p in model_r.backbone.named_parameters() if p.requires_grad], "lr": 2e-4},
            {"params": model_r.head.parameters(), "lr": 5e-3},
        ],
        weight_decay=1e-4,
    )

    for epoch in range(epochs):
        model_r.train()
        for batch in train_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            opt.zero_grad()
            loss = criterion(model_r(pixel_values), labels)
            loss.backward()
            opt.step()

    eval_out = evaluator.evaluate(model_r, val_loader, device=str(device))
    mean_auc: float = float(eval_out["mean_auroc"])
    print(f"Rank r={r} Completed -> Mean AUROC: {mean_auc:.4f}, Trainable: {trainable:,}")

    rec = {"rank": r, "trainable_params": trainable, "param_pct": 100 * trainable / total, "val_auroc": mean_auc}
    sweep_results.append(rec)
    logger.log(
        run_id=f"lora_r{r}_chexpert",
        dataset="CheXpert",
        backbone="DINOv2 ViT-B/14",
        method="LoRA",
        rank=r,
        trainable_params=trainable,
        val_auroc=mean_auc,
    )

## 6. Rank Sweep Results Table

Tabulate rank sweep outcomes and plot the validation AUROC curve against adaptation rank.

In [ ]:
"""
Tabulate rank sweep findings and render validation AUROC versus rank plot.
"""
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

results_df: pd.DataFrame = pd.DataFrame(sweep_results)
print(results_df.to_string(index=False))

sns.set_theme(style="whitegrid")
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.set_xlabel("LoRA Rank (r)", fontsize=12)
ax1.set_ylabel("Validation Mean AUROC", color="tab:blue", fontsize=12)
ax1.plot(results_df["rank"], results_df["val_auroc"], marker="o", linewidth=2.5, color="tab:blue", label="Val AUROC")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.set_ylabel("Trainable Parameters", color="tab:gray", fontsize=12)
ax2.bar(results_df["rank"], results_df["trainable_params"], alpha=0.25, width=2.0, color="tab:gray", label="Params")
ax2.tick_params(axis="y", labelcolor="tab:gray")
ax2.grid(False)

plt.title("E2: LoRA Rank Sweep Performance Trade-Off (CheXpert)", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## 7. Best Checkpoint Selection

Select the optimal LoRA rank configuration on validation data and designate the corresponding checkpoint target.

In [ ]:
"""
Identify optimal rank checkpoint by validation AUROC and designate checkpoint path.
"""
best_row = results_df.loc[results_df["val_auroc"].idxmax()]
best_rank: int = int(best_row["rank"])
print(f"Selected Best Configuration: Rank r={best_rank} with Validation AUROC: {best_row['val_auroc']:.4f}")

best_ckpt_path: str = f"{CHECKPOINT_DIR}/best_lora_r{best_rank}_chexpert.pt"
print(f"Target best checkpoint path: {best_ckpt_path}")